# 01 — Validate paired tracks and their provenance

## Question

Are noisy inputs and projected reference tracks aligned without leaking reference information into inference?

## Inputs

The audit receipt and either an explicitly configured analytic fixture or an audited source TrackBundle. Source preparation, licenses, rendering and extraction are separate prerequisites.

Use an explicit `STV2_CONFIG` JSON file and a unique run ID. The setup resolves relative artifact paths from the repository root. See the [execution guide](README.md), [development protocol](../../docs/studies/synthetic-training-v2/protocol.md) and [literature ledger](../../docs/studies/synthetic-training-v2/literature.md).

In [ ]:
import os
import sys
from pathlib import Path
from IPython.display import Image, display

if not os.environ.get("STV2_CONFIG"):
    raise RuntimeError("Set STV2_CONFIG to an explicit study JSON configuration before execution.")
config_path = Path(os.environ["STV2_CONFIG"]).expanduser().resolve()
if not config_path.is_file():
    raise FileNotFoundError(f"Study configuration does not exist: {config_path}")
search_root = Path(os.environ.get("GAVD6_ROOT", Path.cwd())).expanduser().resolve()
PROJECT_ROOT = next((path for path in (search_root, *search_root.parents)
                     if (path / "src/gavd6_sjepa").is_dir() and (path / "pyproject.toml").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Run inside the repository or set GAVD6_ROOT to its root.")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
os.environ["STV2_CONFIG"] = str(config_path)
from gavd6_sjepa.research_directions.synthetic_training_v2.config import RunConfig
from gavd6_sjepa.research_directions.synthetic_training_v2.workflow import run_stage

cfg = RunConfig.load(os.environ["STV2_CONFIG"])
display({"run_id": cfg.run_id, "mode": cfg.mode, "device": cfg.device,
         "artifact_root": str(cfg.root), "confirmation": "closed"})
if cfg.mode == "fixture":
    print("CPU software fixture: method ordering is not empirical evidence.")

## Computation

Validate the body-12 schema, target validity, observed-input flags, physical timestamps, canonical person groups and held-extractor exposure. Save the validated bundle, achieved counts and an input contact sheet.

`run_stage` implements the computation in the study modules. It checks prerequisite receipts and returns the saved result on an unchanged rerun; a changed configuration or code identity requires a new run ID.

In [ ]:
data = run_stage(cfg, "data", repo_root=PROJECT_ROOT)
display(data)
display(Image(filename=str(cfg.root / "data/track-contact-sheet.png")))

## Outputs and checks

Inspect `data/achieved-size.json`, `data/bundle/` and `data/track-contact-sheet.png`. A contact sheet is a quick input audit, not the independent landmark-overlay or anatomical-convention review required before large source fitting. Keep nuisance pairs, aliases and overlapping source windows together.

Stage receipts under `receipts/` record elapsed time and hashes of produced artifacts. Inspect the saved files for full diagnostics; the display above is deliberately brief.

## Interpretation

Projected SMPL-H centers are approximate synthetic targets. The fixture verifies plumbing and grouping, but cannot establish pose-estimator robustness, anatomical accuracy or a held-family transfer claim. Missing inputs can retain valid restoration targets.

A completed fixture checks software behavior. Scientific gates use `pass`, `fail` or `insufficient_evidence`; fixture success cannot make a scientific gate pass.

## Next gate

Record independent landmark and source-exposure audits before a source screen. Continue to [02 — image adaptation](02_image_adaptation.ipynb) to record the supporting branch, then [03 — information ladder](03_information_ladder.ipynb).